In [4]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, date
import pandas as pd
import re
import time
import random
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import json
import seaborn as sns
from scipy.stats import *
today = datetime.today().strftime('%Y-%m-%d')
today = '2026-06-01'
import json
import os
JSON_CSV_DIR = os.path.join(os.getcwd(), "json_csv")


### Geo data

In [8]:
#Administrative boundaries source: SofiaPlan Open Data.
# Send a request directly to the website API
# c API прямо на сайте  делаем запрос
'''[out:json][timeout:60];

area["name"="София"]->.sofia;

relation(area.sofia)
  ["place"="suburb"];

out ids;'''
# The response contains entries such as: # {'type': 'relation', 'id': 14580016} 
# Save them to JSON, then use these IDs to retrieve district names and geometries
#получаем  типа {'type': 'relation', 'id': 14580016},
#вставляем в json, и потом уже по этим id вытаскиваем name и geometry

from pathlib import Path

file_path1 = os.path.join(JSON_CSV_DIR, "id_districts.json")

with open(file_path1, encoding="utf-8") as f:
    district_id_data = json.load(f)
osm_ids = [ row["id"]  for row in district_id_data]

In [10]:
import requests

OVERPASS_URL = "https://overpass-api.de/api/interpreter"


def get_relation_tags(osm_id):
    query = f"""
    [out:json][timeout:60];
    relation({osm_id});
    out tags;
    """
    response = requests.post(
        OVERPASS_URL,
        data={"data": query},
        headers={"User-Agent": "sofia-real-estate-student-project/1.0"}, timeout=(10, 60)
    )
    print("Status:", response.status_code)

    if response.status_code != 200:
        print(response.text[:1000])
    response.raise_for_status()
    data = response.json()
    return data["elements"][0]["tags"]

In [ ]:
# First, extract district names
#вытаскиваем сначала названия районов
file_path2 = os.path.join(JSON_CSV_DIR, "id_name_districts.json")

if os.path.exists(file_path2):
    with open(file_path2, encoding="utf-8") as f:
        rows = json.load(f)

    print(f"Загружено из файла: {len(rows)}")
else:
    rows = []


    
while True:
    done_ids = {row["osm_id"] for row in rows}
    last_ids = list(set(osm_ids) - done_ids)

    if len(last_ids) == 0:
        print("Все данные успешно загружены")
        break

    print(f"Осталось загрузить: {len(last_ids)}")

    for i, osm_id in enumerate(last_ids):
        try:
            tags = get_relation_tags(osm_id)

            rows.append({
                "osm_id": osm_id,
                "name": tags.get("name"),
                "name_en": tags.get("name:en"),
                "place": tags.get("place"),
                "wikidata": tags.get("wikidata"),
                "wikipedia": tags.get("wikipedia")
            })

            print(f"{i+1}/{len(last_ids)} - OK - {tags.get('name')}")
            time.sleep(2)

        except Exception as e:
            print(f"{i+1}/{len(last_ids)} - ERROR - {osm_id}")
            print(e)
            time.sleep(10)

Загружено из файла: 117
Все данные успешно загружены


In [ ]:
import os
file_path2 = os.path.join(JSON_CSV_DIR, "id_name_districts.json")

if not os.path.exists(file_path2):
    with open(file_path2, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
else:
    print(f"Файл уже существует: {file_path2}")

Файл уже существует: /home/kuspus/Documents/simulative/MathAndStatistics/math_statist_project/json_csv/id_name_districts.json


In [6]:
# Next, extract the geometry for each district
#Теперь вытаскиваем геометрию каждого квартала
def get_geometry_by_osm_id(osm_id):
    query = f"""
[out:json][timeout:120];

relation({osm_id});

out geom;
"""
    response = requests.post(
        OVERPASS_URL,
        data={"data": query},
        headers={"User-Agent": "sofia-real-estate-student-project/1.0"},     timeout=(10, 60)
    )
    print("Status:", response.status_code)

    if response.status_code != 200:
        print(response.text[:1000])

    response.raise_for_status()
    data = response.json()

    if not data.get("elements"):
        return None
    return data["elements"][0]


In [11]:
file_path3 = os.path.join(JSON_CSV_DIR, "id_geom_districts.json")

if os.path.exists(file_path3):
    with open(file_path3, "r", encoding="utf-8") as f:
        all_geometries = json.load(f)["elements"]

    print(f"Загружено геометрий из файла: {len(all_geometries)}")

else:
    all_geometries = []
    
while True:

    done_ids = {g["id"] for g in all_geometries}
    last_geoms = list(set(osm_ids) - done_ids)

    if len(last_geoms) == 0:
        print("Все геометрии успешно загружены")
        break

    print(f"Осталось загрузить: {len(last_geoms)}")

    for i, osm_id in enumerate(last_geoms):
        try:
            geometry = get_geometry_by_osm_id(osm_id)
            if geometry is not None:
                all_geometries.append(geometry)
                name = geometry.get("tags", {}).get("name")
                print(
                    f"{i+1}/{len(last_geoms)} "
                    f"OK: {name}"
                )
            else:
                print(
                    f"{i+1}/{len(last_geoms)} "
                    f"EMPTY: {osm_id}"
                )
            time.sleep(6)
        except Exception as e:
            print(
                f"{i+1}/{len(last_geoms)} "
                f"ERROR: {osm_id}"
            )
            print(e)
            time.sleep(30)
    with open(file_path3, "w", encoding="utf-8") as f:
        json.dump({"elements": all_geometries},f, ensure_ascii=False,indent=2)

    print(f"Промежуточно сохранено. Геометрий: {len(all_geometries)}")

Загружено геометрий из файла: 111
Осталось загрузить: 6
Status: 200
1/6 OK: ж.к. Белите брези
Status: 200
2/6 OK: ж.к. Малинова долина
Status: 200
3/6 OK: ж.к. Дианабад
Status: 200
4/6 OK: кв. Обеля
Status: 200
5/6 OK: НПЗ Дианабад
Status: 200
6/6 OK: ж.к. Света Троица
Промежуточно сохранено. Геометрий: 117
Все геометрии успешно загружены


In [ ]:
# The id_geom_districts.json file was converted into districts_from_overpass.geojson (AI)/ Файл id_geom_districtsю.json переделан в districts_from_overpass.geojson (AI)
# The data was reviewed in geojson.io: / Эти данные были проверены в geojson.io:
# missing districts were added, individual polygons were corrected,/ добавлены отсутствующие кварталы, исправлены отдельные полигоны,
# duplicates and unnamed objects were removed./ удалены дубликаты и объекты без названий. 
#The final map was saved in GeoJSON format as map.geojson / Итоговая карта сохранена в формате GeoJSON. map.geojson
#id_districts.json -> id_name_districts.json  -> id_geom_districts.json -> (AI)districts_from_overpass.geojson -> (manual corrections)map.geojson

#Manual corrections in geojson.io: / Правки руками в geojson.io:
#districts_clean = [re.sub(r"^(ж\.к\.|кв\.|в\.з\.|НПЗ|СПЗ)\s*", "", x['name']).strip()  for x in rows]
#'Горна баня' - 2 objects, Дианабад - 2 objects, Доужба  - only 1 object (instead of Дружба 1, Друбта 2), Изток - 2, Люлин   - only 1(instead of 4,7), Малинова Додлина - 2 objects, 
# There are 'Манастирски ливади - Б' and 'Манастирски ливади",   'Младост 1'and 'Младост 1А', Оборище is missing, Овча купел - 2 objects( while imot.bg uses 'Овча купел'and 'Овча купел 1',and 'Овча купел 2'), 
# Редута - is missing, 'Хаджи Димитър'- 2 objects